# Executive Summary — Bank Transaction Fraud Detection
## End-to-End Unsupervised Anomaly Detection Pipeline

---

### Project Overview

| Item | Detail |
|------|--------|
| **Dataset** | 50,000 synthetic bank transactions, 495 accounts, 43 US cities (2020–2025) |
| **Objective** | Detect fraudulent transactions without labeled data using unsupervised ML |
| **Approach** | EDA → Feature Engineering (30 features) → 3 ML Models → Rule Engine → Hybrid Scoring |
| **Key Result** | 190 consensus anomalies identified, 94.7% caught by rule engine |

---

## Pipeline Architecture

```
Raw Data (50K txns)
    │
    ├── 01. EDA ──────────────────── Data quality, distributions, anomaly indicators
    │
    ├── 02. Feature Engineering ──── 30 derived features (account, transaction, device, time)
    │       │
    │       ├── Isolation Forest ─── 2,500 anomalies (5.0%)
    │       ├── DBSCAN ──────────── 24 anomalies (0.05%)
    │       └── LOF ─────────────── 2,500 anomalies (5.0%)
    │
    ├── 03. Model Evaluation ─────── Internal metrics, agreement analysis, significance tests
    │
    ├── 04. Rule-Based Engine ────── 7 expert rules + Hybrid Score (ML + Rules)
    │
    └── 05. Final Report ─────────── This notebook (Executive Summary)
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize':(16,6),'font.size':12,'axes.titleweight':'bold'})

# ── Full Pipeline (condensed) ──
df = pd.read_csv('bank_transactions.csv')
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], format='mixed')

# Features
acct = df.groupby('AccountID').agg(
    acct_txn_count=('TransactionID','count'),acct_mean_amount=('TransactionAmount','mean'),
    acct_std_amount=('TransactionAmount','std'),acct_max_amount=('TransactionAmount','max'),
    acct_mean_duration=('TransactionDuration','mean'),acct_mean_login=('LoginAttempts','mean'),
    acct_unique_devices=('DeviceID','nunique'),acct_unique_locations=('Location','nunique'),
).reset_index()
acct['acct_std_amount']=acct['acct_std_amount'].fillna(0)
df=df.merge(acct,on='AccountID',how='left')
df['amount_zscore']=(df['TransactionAmount']-df['acct_mean_amount'])/df['acct_std_amount'].replace(0,1)
df['amount_to_balance_ratio']=df['TransactionAmount']/df['AccountBalance'].replace(0,1)
df['amount_to_max_ratio']=df['TransactionAmount']/df['acct_max_amount'].replace(0,1)
df['duration_deviation']=abs(df['TransactionDuration']-df['acct_mean_duration'])
df['high_login_flag']=(df['LoginAttempts']>=3).astype(int)
Q1,Q3=df['TransactionAmount'].quantile(0.25),df['TransactionAmount'].quantile(0.75)
df['amount_outlier_flag']=(df['TransactionAmount']>Q3+1.5*(Q3-Q1)).astype(int)
dev=df.groupby('DeviceID')['AccountID'].nunique().reset_index();dev.columns=['DeviceID','device_shared_accounts']
df=df.merge(dev,on='DeviceID',how='left');df['device_shared_flag']=(df['device_shared_accounts']>1).astype(int)
ip=df.groupby('IP Address')['AccountID'].nunique().reset_index();ip.columns=['IP Address','ip_shared_accounts']
df=df.merge(ip,on='IP Address',how='left')
mf=df.groupby(['AccountID','MerchantID']).size().reset_index(name='merchant_visit_count')
df=df.merge(mf,on=['AccountID','MerchantID'],how='left');df['is_new_merchant']=(df['merchant_visit_count']==1).astype(int)
df['is_weekend']=(df['TransactionDate'].dt.dayofweek>=5).astype(int)
df['txn_date']=df['TransactionDate'].dt.date
dv=df.groupby(['AccountID','txn_date']).size().reset_index(name='daily_txn_count')
df=df.merge(dv,on=['AccountID','txn_date'],how='left')
df=df.sort_values(['AccountID','TransactionDate'])
df['time_since_last_txn']=df.groupby('AccountID')['TransactionDate'].diff().dt.total_seconds()/3600
df['time_since_last_txn']=df['time_since_last_txn'].fillna(-1)
df['rapid_txn_flag']=((df['time_since_last_txn']>=0)&(df['time_since_last_txn']<1)).astype(int)

# Rules
df['R1_high_login']=(df['LoginAttempts']>=3).astype(int)
df['R2_amount_spike']=(df['amount_zscore']>2).astype(int)
df['R3_high_ratio']=(df['amount_to_balance_ratio']>0.8).astype(int)
df['R4_device_sharing']=(df['device_shared_accounts']>=5).astype(int)
df['R5_rapid_txn']=(df['rapid_txn_flag']==1).astype(int)
df['R6_multi_location']=(df['acct_unique_locations']>=8).astype(int)
df['R7_high_velocity']=(df['daily_txn_count']>=3).astype(int)
rcols=[c for c in df.columns if c.startswith('R') and c[1].isdigit()]
wts={'R1_high_login':3,'R2_amount_spike':2.5,'R3_high_ratio':2,'R4_device_sharing':2.5,'R5_rapid_txn':1.5,'R6_multi_location':2,'R7_high_velocity':1.5}
df['rule_score']=sum(df[r]*wts[r] for r in rcols)/sum(wts.values())
df['rules_triggered']=df[rcols].sum(axis=1)

# ML Models
feats=['TransactionAmount','TransactionDuration','LoginAttempts','AccountBalance','amount_zscore','amount_to_balance_ratio','amount_to_max_ratio','duration_deviation','high_login_flag','amount_outlier_flag','device_shared_accounts','device_shared_flag','ip_shared_accounts','is_new_merchant','acct_txn_count','acct_std_amount','acct_unique_devices','acct_unique_locations','daily_txn_count','rapid_txn_flag','is_weekend','CustomerAge']
X=df[feats].replace([np.inf,-np.inf],np.nan).fillna(0)
Xs=pd.DataFrame(StandardScaler().fit_transform(X),columns=feats,index=X.index)
Xp=PCA(n_components=5,random_state=42).fit_transform(Xs)
iso=IsolationForest(n_estimators=200,contamination=0.05,random_state=42,n_jobs=-1)
df['iso_anomaly']=(iso.fit_predict(Xs)==-1).astype(int);df['iso_score']=iso.decision_function(Xs)
df['db_anomaly']=(DBSCAN(eps=2.5,min_samples=10,n_jobs=-1).fit_predict(Xp)==-1).astype(int)
lof=LocalOutlierFactor(n_neighbors=20,contamination=0.05,n_jobs=-1)
df['lof_anomaly']=(lof.fit_predict(Xs)==-1).astype(int);df['lof_score']=-lof.negative_outlier_factor_
df['ml_score']=(0.45*MinMaxScaler().fit_transform(-df[['iso_score']]).flatten()+0.35*MinMaxScaler().fit_transform(df[['lof_score']]).flatten()+0.20*df['db_anomaly'].values)
df['hybrid_score']=0.5*df['ml_score']+0.5*df['rule_score']
df['hybrid_level']=pd.cut(df['hybrid_score'],bins=[0,0.1,0.25,0.45,1.0],labels=['Low','Medium','High','Critical'],include_lowest=True)
df['models_flagged']=df['iso_anomaly']+df['db_anomaly']+df['lof_anomaly']

print(f'Pipeline complete: {len(df):,} transactions processed')
print(f'Features: {len(feats)} | Rules: {len(rcols)} | Models: 3')

---
## Key Findings Dashboard

In [ ]:
# ── Final Dashboard ──
fig = plt.figure(figsize=(22, 14))
gs = gridspec.GridSpec(3, 4, hspace=0.45, wspace=0.35)

# Row 1: Risk Distribution
ax1 = fig.add_subplot(gs[0, 0])
risk_vc = df['hybrid_level'].value_counts().reindex(['Low','Medium','High','Critical'])
ax1.pie(risk_vc.values, labels=risk_vc.index, colors=['#22c55e','#f59e0b','#ef4444','#7f1d1d'],
        autopct='%1.1f%%', textprops={'fontsize':10}, startangle=90)
ax1.set_title('Hybrid Risk Distribution')

# Model agreement
ax2 = fig.add_subplot(gs[0, 1])
flag_vc = df['models_flagged'].value_counts().sort_index()
ax2.bar(flag_vc.index, flag_vc.values, color=['#22c55e','#fbbf24','#f97316','#ef4444'][:len(flag_vc)], edgecolor='black', linewidth=0.5)
ax2.set_title('ML Model Agreement')
ax2.set_xlabel('Models Flagged'); ax2.set_ylabel('Count')
ax2.set_xticks(flag_vc.index)

# Rules triggered
ax3 = fig.add_subplot(gs[0, 2])
rule_counts = pd.Series({r.replace('_',' '): df[r].sum() for r in rcols}).sort_values()
rule_counts.plot(kind='barh', ax=ax3, color='#6b3a2a', edgecolor='black', linewidth=0.5)
ax3.set_title('Rule Trigger Frequency')

# Channel risk
ax4 = fig.add_subplot(gs[0, 3])
ch = df.groupby('Channel')['hybrid_score'].mean().sort_values(ascending=False)
ch.plot(kind='bar', ax=ax4, color=['#ef4444','#f59e0b','#3b82f6'], edgecolor='black', linewidth=0.5)
ax4.set_title('Avg Hybrid Risk by Channel')
ax4.tick_params(axis='x', rotation=0)

# Row 2: ML vs Rules scatter
ax5 = fig.add_subplot(gs[1, :2])
scatter = ax5.scatter(df['ml_score'], df['rule_score'], c=df['hybrid_score'],
                      cmap='YlOrRd', alpha=0.15, s=5, rasterized=True)
ax5.set_title('ML Score vs Rule Score (color = Hybrid)')
ax5.set_xlabel('ML Score'); ax5.set_ylabel('Rule Score')
plt.colorbar(scatter, ax=ax5, label='Hybrid Score', shrink=0.8)

# Hybrid score distribution
ax6 = fig.add_subplot(gs[1, 2:])
ax6.hist(df['hybrid_score'], bins=60, color='steelblue', edgecolor='white', alpha=0.7)
for t, l, c in [(0.1,'Low|Med','#22c55e'),(0.25,'Med|High','#f59e0b'),(0.45,'High|Crit','#ef4444')]:
    ax6.axvline(t, color=c, linestyle='--', linewidth=2, label=f'{l} ({t})')
ax6.set_title('Hybrid Risk Score Distribution')
ax6.set_xlabel('Score'); ax6.legend(fontsize=8)

# Row 3: Full-width amount vs risk
ax7 = fig.add_subplot(gs[2, :])
scatter2 = ax7.scatter(df['TransactionAmount'], df['hybrid_score'],
                       c=df['rules_triggered'], cmap='RdYlGn_r', alpha=0.12, s=5, rasterized=True)
ax7.axhline(0.25, color='#f59e0b', linestyle='--', alpha=0.5, label='High threshold')
ax7.axhline(0.45, color='#ef4444', linestyle='--', alpha=0.5, label='Critical threshold')
ax7.set_title('Transaction Amount vs Hybrid Risk Score')
ax7.set_xlabel('Amount ($)'); ax7.set_ylabel('Hybrid Score')
ax7.legend(loc='upper right')
plt.colorbar(scatter2, ax=ax7, label='Rules Triggered', shrink=0.6)

plt.suptitle('BankGuard Analytics — Final Report Dashboard', fontsize=20, fontweight='bold', y=1.01)
plt.show()

---
## Summary Statistics

In [ ]:
# ── Final Summary Print ──
n = len(df)
ml_consensus = (df['models_flagged']>=2).sum()
all3 = (df['models_flagged']==3).sum()
any_rule = (df['rules_triggered']>0).sum()
high_crit = df['hybrid_level'].isin(['High','Critical']).sum()

print('='*70)
print('     BANKGUARD ANALYTICS — EXECUTIVE SUMMARY')
print('='*70)
print()
print('  DATA OVERVIEW')
print(f'    Transactions:          {n:>8,}')
print(f'    Accounts:              {df["AccountID"].nunique():>8,}')
print(f'    Locations:             {df["Location"].nunique():>8,}')
print(f'    Date Range:            2020 — 2025')
print(f'    Total Volume:          ${df["TransactionAmount"].sum():>12,.0f}')
print()
print('  FEATURE ENGINEERING')
print(f'    Features Created:      {len(feats):>8}')
print(f'    Feature Groups:        4 (Account, Transaction, Device, Time)')
print()
print('  ML MODELS')
print(f'    Isolation Forest:      {df["iso_anomaly"].sum():>8,} anomalies (5.0%)')
print(f'    DBSCAN:                {df["db_anomaly"].sum():>8,} anomalies ({df["db_anomaly"].mean()*100:.1f}%)')
print(f'    LOF:                   {df["lof_anomaly"].sum():>8,} anomalies (5.0%)')
print(f'    ML Consensus (2+):     {ml_consensus:>8,} ({ml_consensus/n*100:.1f}%)')
print(f'    All 3 Agree:           {all3:>8,} ({all3/n*100:.2f}%)')
print()
print('  RULE ENGINE')
print(f'    Rules Defined:         {len(rcols):>8}')
print(f'    Transactions Flagged:  {any_rule:>8,} ({any_rule/n*100:.1f}%)')
print(f'    Rule Recall (ML 2+):        94.7%')
print()
print('  HYBRID RISK SCORING')
for lv in ['Low','Medium','High','Critical']:
    c = (df['hybrid_level']==lv).sum()
    print(f'    {lv:10s}:            {c:>8,} ({c/n*100:.1f}%)')
print(f'    High + Critical:       {high_crit:>8,} ({high_crit/n*100:.1f}%)')
print()
print('  TOP RISK DRIVERS')
print(f'    #1 Login ≥ 3:          +164% risk premium')
print(f'    #2 Amount-to-Balance:  +527% in anomalies')
print(f'    #3 Amount Z-Score:     +3,704% in anomalies')
print(f'    #4 Daily Velocity:     +79% in anomalies')
print()
print('  RECOMMENDATIONS')
print('    1. Deploy hybrid scoring in production (ML + Rules)')
print('    2. Set automated alerts for Login ≥ 3 + Amount spike')
print('    3. Implement device fingerprinting')
print('    4. Add real-time velocity checks')
print('    5. Build feedback loop for analyst validation')
print()
print('='*70)

---
## Notebook Index

| # | Notebook | Description |
|---|----------|-------------|
| 01 | `01_EDA_Bank_Transactions.ipynb` | Exploratory Data Analysis |
| 02 | `02_Feature_Engineering_and_Anomaly_Detection.ipynb` | 30 Features + 3 ML Models |
| 03 | `03_Model_Evaluation_and_Comparison.ipynb` | Internal Metrics + Sensitivity Analysis |
| 04 | `04_Rule_Based_Engine.ipynb` | 7 Expert Rules + Hybrid Scoring |
| 05 | `05_Final_Report.ipynb` | Executive Summary (this notebook) |
| — | `dashboard.py` | Interactive Streamlit Dashboard |
| — | `presentation.html` | Web Presentation (GitHub Pages) |

---

*BankGuard Analytics — Fraud Detection Pipeline*